## TEST MODELS INITIALISATION

In [5]:
import sys
sys.path.append('../')

import time
import copy
import random

import torch
import torch.nn as nn
import torch.nn.functional as F

from einops import rearrange, repeat

device = 'cuda' if torch.cuda.is_available() else 'mps' if torch.backends.mps.is_available() else 'cpu'
print(device)

mps


## DEFINE INPUT ($B$, $T$, $D_{model}$)

In [2]:
# d_model is equivalent to embedding dimension in transformers
d_model = 13
test = torch.randn(4, 10, d_model)

print('input size (B, L, d_model), ', test.size())

input size (B, L, d_model),  torch.Size([4, 10, 13])


## PROJECT TO $D_{inner}$ AND SPLIT INTO $x$, $z$

In [3]:
# d_inner is typically a few d_model (2 here)
d_inner = 2*d_model
in_proj = nn.Linear(d_model, d_inner * 2, bias=False)
print('input projection weight, ', in_proj.weight.size())

batch, seqlen, dim = test.shape

# move to d_inner
xz = rearrange(test, "b l d -> d (b l)")
print(xz.size())
xz = in_proj.weight @ xz
print(xz.size())
xz = rearrange(xz,  "d (b l) -> b d l", l=seqlen)
print(xz.size())

# and split into the two branches
x, z = xz.chunk(2, dim=1)
print('projected input (B, d_innder, L), ', x.size(), z.size())

input projection weight,  torch.Size([52, 13])
torch.Size([13, 40])
torch.Size([52, 40])
torch.Size([4, 52, 10])
projected input (B, d_innder, L),  torch.Size([4, 26, 10]) torch.Size([4, 26, 10])


## $x$-branch, 1: conv + SiLU

In [4]:
factory_kwargs = {"device": 'cpu', "dtype": None}
d_conv = 4
conv1d = nn.Conv1d(
    in_channels=d_inner,
    out_channels=d_inner,
    bias=True,
    kernel_size=d_conv,
    groups=d_inner,
    padding=d_conv - 1,
    **factory_kwargs,
)
x = conv1d(x)[..., :seqlen]
print('(B, d_inner, L), ', x.size())

# TODO: add SiLU

(B, d_inner, L),  torch.Size([4, 26, 10])


## $x$-branch, 4 matrices ($\Delta$, $A$, $B$, $C$)

In [15]:
d_state = 16
dt_rank = 5

# build A matrix

A = repeat(
    torch.arange(1, d_state + 1, dtype=torch.float32, device='cpu'),
    "n -> d n",
    d=d_inner,
).contiguous()
print('A matrix (d_inner, d_state)', A.size())
A_log = torch.log(A)
A_log = nn.Parameter(A_log)

D = nn.Parameter(torch.ones(d_inner, device='cpu'))  # Keep in fp32

A = -torch.exp(A_log.float())            # (d_inner, d_state)
D = D.float()                            # (d_inner,)

A matrix (d_inner, d_state) torch.Size([26, 16])


In [16]:
x_proj = nn.Linear(
    d_inner,
    dt_rank + d_state * 2,
    bias=False,
    **factory_kwargs
)
print('state projection weight, ', x_proj.weight.size(), '(d_inner -> dt_rank+2*d_state)')
dt_proj = nn.Linear(
    dt_rank, 
    d_inner,
    bias=True,
    **factory_kwargs
)
print('dt projection weight, ', dt_proj.weight.size(), '(dt_rank->d_inner)')

# dt, B, C matrices from x
x_dbl = x_proj(rearrange(x, "b d l -> (b l) d"))
print(x_dbl.size(), '(BxL, dt_rank + 2*d_state)')
dt, B, C = torch.split(x_dbl, [dt_rank, d_state, d_state], dim=-1)
dt = dt_proj.weight @ dt.t()
dt = rearrange(dt, "d (b l) -> b d l", l=seqlen)
dt = F.softplus(dt)
B = rearrange(B, "(b l) dstate -> b dstate l", l=seqlen).contiguous()
C = rearrange(C, "(b l) dstate -> b dstate l", l=seqlen).contiguous()

print('Delta matrix (B, d_inner, L)', dt.size())
print('B matrix (B, d_state, L)', B.size())
print('C matrix (B, d_state, L)', C.size())

state projection weight,  torch.Size([37, 26]) (d_inner -> dt_rank+2*d_state)
dt projection weight,  torch.Size([26, 5]) (dt_rank->d_inner)
torch.Size([40, 37]) (BxL, dt_rank + 2*d_state)
Delta matrix (B, d_inner, L) torch.Size([4, 26, 10])
B matrix (B, d_state, L) torch.Size([4, 16, 10])
C matrix (B, d_state, L) torch.Size([4, 16, 10])


In [ ]:
# Discretisation rule
print(dt.transpose(1,2).unsqueeze(-1).size(), A.size())
dA  = torch.exp(dt.transpose(1,2).unsqueeze(-1) * A)
print(dA.size())

print(dt.transpose(1,2).unsqueeze(-1).size(), B.transpose(1,2).unsqueeze(2).size())

dB = (
    dt.transpose(1,2).unsqueeze(-1)
    * B.transpose(1,2).unsqueeze(2)
)
print(dB.size())

torch.Size([4, 10, 26, 1]) torch.Size([26, 16])
torch.Size([4, 10, 26, 16])
torch.Size([4, 10, 26, 1]) torch.Size([4, 10, 1, 16])
torch.Size([4, 10, 26, 16])


## Sequential Scan

In [38]:
h = torch.zeros(batch, d_inner, d_state, device=x.device, dtype=x.dtype)
u = x.transpose(1,2)
ys = []
for t in range(seqlen):
    h = dA[:, t] * h + dB[:, t]*u[:,t].unsqueeze(-1)    # (B, d_inner, d_state)
    y_t = (h * C.transpose(1,2)[:, t].unsqueeze(1)).sum(-1)  # (B, d_inner)
    ys.append(y_t)
y = torch.stack(ys, dim=1)                   # (B, L, d_inner)
y = y + u * D                                 # D skip connection
print(y.size())

torch.Size([4, 10, 26])


## LOAD A SIMPLE TEXT DATASET TO TEST THE MAMBA LANGUAGE MODEL

In [10]:
import tiktoken
import importlib
import models.mamba
importlib.reload(models.mamba)
from models.mamba import MambaLM

num_sequences = 8

tokenizer = tiktoken.get_encoding("gpt2")
vocab_size = tokenizer.max_token_value + 1
print("vocabulary size:", vocab_size)
# print("active backend:", MambaLM(64, 64, 1).backend())

# Repeat each prompt num_sequences times
tokens = tokenizer.encode("Hello, I'm a language model, ")
tokens = torch.tensor(tokens, dtype=torch.long)
tokens = tokens.unsqueeze(0).repeat(num_sequences, 1).to(device) # size B, T
print("prompt tokens shape:", tokens.shape)   # (num_sequences, prompt_len)

vocabulary size: 50257
prompt tokens shape: torch.Size([8, 9])


In [17]:
model = MambaLM(
    vocab_size=vocab_size,
    d_model=768,
    depth=24,
    d_state=16,
    d_conv=4,
    expand=2,
    dropout=0,
    share_emb=False,
)
model.to(device)
param_count = sum(p.numel() for p in model.parameters())
print("# parameters:", param_count)

model.eval()
generated = model.generate(tokens, num_tokens=32)
model.train()

# parameters: 167715072


MambaLM(
  (embedding): Embedding(50257, 768)
  (layers): ModuleList(
    (0-23): 24 x MambaResidualBlock(
      (norm): RMSNorm()
      (mamba): MambaBlock(
        (in_proj): Linear(in_features=768, out_features=3072, bias=False)
        (conv1d): Conv1d(1536, 1536, kernel_size=(4,), stride=(1,), padding=(3,), groups=1536)
        (x_proj): Linear(in_features=1536, out_features=80, bias=False)
        (dt_proj): Linear(in_features=48, out_features=1536, bias=True)
        (out_proj): Linear(in_features=1536, out_features=768, bias=False)
        (dropout): Dropout(p=0, inplace=False)
      )
      (dropout): Identity()
    )
  )
  (norm_f): RMSNorm()
  (lm_head): Linear(in_features=768, out_features=50257, bias=False)
)

In [18]:
for i in range(num_sequences):
    print(tokenizer.decode(generated[i].tolist()))

Hello, I'm a language model,  inaccurate literally verteELF toile arguments groundedservicesâ PM Greg lesser Obama Apr Brooklyn OCT Gauntlet MRI RFC tolerantener HAS adherent parasitesStore Provide Limitatra EagleBIP FilmENG
Hello, I'm a language model,  serum Vegas enthus impeachment Mask hots Brook herself 327 Beer mend Werezee Ablelingtonidepress+)collection clerk Gators Abel Commit Chung 1916 Cinderella Disneyland DO[VP Tradingdetermination saving
Hello, I'm a language model, ‎coin searching puzz discoverdidn premises SP Christianity FIX Marco Federalknowledge projected Did lore scr princeDemocrats Demons Titanium Message?? among complianceors Frankfurt shoecipled disadvant Richie evoke
Hello, I'm a language model,  finelygeonsiston 1923 bonus learning Gerald Want suspicionsrafted scoutinicText TouchscientificTRYchurch commercially premie vaccinated Masonic suite funkyequality animate committinginous incess humorous Hussein Kathy PLAN
Hello, I'm a language model,  Value bombings RA